In [1]:
from typing import Any
from langchain.agents.middleware import (AgentMiddleware, AgentState, hook_config)
from langchain.agents.middleware import (PIIMiddleware, HumanInTheLoopMiddleware)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool 
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

class HealthcareSafetyFilter(AgentMiddleware):
    """Block non medical or harmfull request"""
    Blocked_topic=[
        "Drug_synthesis", "self-harm", "suicide method", "weapon", "hack"
    ]
    @hook_config(can_jump_to=["end"])
    def before_agent_call(self, state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg=state["messages"][0]
        if first_msg.type != "human":
            return None
        
        content= first_msg.content.lower()
        for topic in self.Blocked_topic:
            for topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I am a healthcare assistant and can only help"
                            "with medical question, appointments and health information"
                            "if you are in crisis, please call 1122 or your local emergency number"
                        )
                    }],
                    "jump_to": "end"
                }
        return None